# Fundus Vascular Bifurcation Keypoint Detection with SuperRetina

This notebook covers the full pipeline:
1. Environment setup & data mounting
2. GT label generation (vessel segmentation → skeletonization → bifurcation extraction)
3. Dataset preparation
4. SuperRetina model training
5. Inference & visualization
6. Vessel network analysis (node count, vessel length, vessel number)
7. Learning curve plotting

## 1. Environment Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install required packages
!pip install scikit-image opencv-python-headless albumentations tqdm networkx -q

In [ ]:
# Clone SuperRetina (only needed if not already in Drive)
import os

REPO_DIR = '/content/SuperRetina'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/ruc-aimc-lab/SuperRetina.git {REPO_DIR}

os.chdir(REPO_DIR)
import sys
sys.path.insert(0, REPO_DIR)

In [ ]:
import json
import random
import glob
import copy

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn import functional as F

from skimage.morphology import skeletonize
from skimage.filters import frangi
import networkx as nx

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 2. Data Paths Configuration

Upload the dataset folders to your Google Drive and set the paths below.
Expected structure in Drive:
```
MyDrive/
  data1/DIARETDB1/, DRIONS-DB/, Drishti-GS/
  data2/CHASEDB1/, DRIVE/, e-ophtha/, FIRE/, HRF/, MESSIDOR/
```

In [ ]:
# Update these paths to match your Google Drive structure
DRIVE_ROOT = '/content/drive/MyDrive'

DATASET_DIRS = {
    'DIARETDB1': f'{DRIVE_ROOT}/data1/DIARETDB1',
    'DRIONS-DB': f'{DRIVE_ROOT}/data1/DRIONS-DB',
    'Drishti-GS': f'{DRIVE_ROOT}/data1/Drishti-GS',
    'CHASEDB1':  f'{DRIVE_ROOT}/data2/CHASEDB1',
    'DRIVE':     f'{DRIVE_ROOT}/data2/DRIVE',
    'e-ophtha':  f'{DRIVE_ROOT}/data2/e-ophtha',
    'FIRE':      f'{DRIVE_ROOT}/data2/FIRE',
    'HRF':       f'{DRIVE_ROOT}/data2/HRF',
    'MESSIDOR':  f'{DRIVE_ROOT}/data2/MESSIDOR',
}

OUTPUT_DIR = '/content/superretina_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/gt_labels', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/results', exist_ok=True)

In [ ]:
def collect_image_paths(dataset_dirs, extensions=('*.jpg', '*.png', '*.ppm', '*.tif', '*.bmp')):
    """Recursively collect all fundus image paths from dataset directories."""
    all_paths = []
    for name, base in dataset_dirs.items():
        if not os.path.exists(base):
            print(f'[WARN] Not found: {base}')
            continue
        found = []
        for ext in extensions:
            found += glob.glob(os.path.join(base, '**', ext), recursive=True)
        # Exclude mask / annotation files heuristically
        found = [p for p in found if 'mask' not in p.lower() and
                 'label' not in p.lower() and 'annot' not in p.lower() and
                 'gt' not in os.path.basename(p).lower()]
        all_paths += [(p, name) for p in found]
        print(f'{name}: {len(found)} images')
    print(f'Total: {len(all_paths)} images')
    return all_paths

all_image_paths = collect_image_paths(DATASET_DIRS)

## 3. GT Label Generation Pipeline

Pipeline: Frangi filter → binary vessel mask → skeletonization → bifurcation point extraction

In [ ]:
def preprocess_fundus(img_bgr, target_size=512):
    """Convert fundus image to grayscale green channel and resize."""
    if img_bgr is None:
        return None
    # Green channel is richest in retinal vessel contrast
    green = img_bgr[:, :, 1]
    # CLAHE enhancement
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(green)
    resized = cv2.resize(enhanced, (target_size, target_size))
    return resized


def segment_vessels_frangi(gray, thresh_percentile=85):
    """Vessel segmentation using Frangi filter."""
    gray_norm = gray.astype(np.float32) / 255.0
    vessel_map = frangi(gray_norm, sigmas=range(1, 4), black_ridges=False)
    vessel_map = (vessel_map - vessel_map.min()) / (vessel_map.max() - vessel_map.min() + 1e-8)
    thresh = np.percentile(vessel_map, thresh_percentile)
    binary = (vessel_map > thresh).astype(np.uint8)
    # Morphological cleanup
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    return binary


def extract_bifurcations(skeleton):
    """Extract bifurcation points: skeleton pixels with 3+ neighbors."""
    kernel = np.ones((3, 3), dtype=np.uint8)
    neighbor_count = cv2.filter2D(skeleton.astype(np.uint8), -1, kernel)
    # A skeleton pixel is a bifurcation if it has 3 or more neighbors (including itself)
    bifurcation_map = (skeleton > 0) & (neighbor_count >= 4)  # >=4 means 3+ neighbors
    ys, xs = np.where(bifurcation_map)
    points = list(zip(xs.tolist(), ys.tolist()))  # (x, y)
    return points, bifurcation_map.astype(np.uint8)


def generate_gt_label(img_path, target_size=512, min_points=5):
    """Full pipeline: image → bifurcation keypoints."""
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        return None, None, None
    gray = preprocess_fundus(img_bgr, target_size)
    if gray is None:
        return None, None, None
    vessel_mask = segment_vessels_frangi(gray)
    skeleton = skeletonize(vessel_mask > 0).astype(np.uint8)
    points, bifurcation_map = extract_bifurcations(skeleton)
    if len(points) < min_points:
        return gray, vessel_mask, None
    return gray, vessel_mask, points


print('GT generation functions defined.')

In [ ]:
# Visualize the pipeline on one sample image
if all_image_paths:
    sample_path, sample_dataset = all_image_paths[0]
    img_bgr = cv2.imread(sample_path)
    gray = preprocess_fundus(img_bgr, 512)
    vessel_mask = segment_vessels_frangi(gray)
    skeleton = skeletonize(vessel_mask > 0).astype(np.uint8)
    points, bif_map = extract_bifurcations(skeleton)

    fig, axes = plt.subplots(1, 5, figsize=(22, 4))
    axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Original Fundus')
    axes[1].imshow(gray, cmap='gray')
    axes[1].set_title('Green Ch + CLAHE')
    axes[2].imshow(vessel_mask, cmap='gray')
    axes[2].set_title('Vessel Mask (Frangi)')
    axes[3].imshow(skeleton, cmap='gray')
    axes[3].set_title('Skeleton')
    axes[4].imshow(gray, cmap='gray')
    if points:
        xs, ys = zip(*points)
        axes[4].scatter(xs, ys, s=3, c='red', linewidths=0)
    axes[4].set_title(f'Bifurcations ({len(points)})')
    for ax in axes:
        ax.axis('off')
    plt.suptitle(f'GT Pipeline - {sample_dataset}', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/results/pipeline_sample.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Bifurcation points found: {len(points)}')

In [ ]:
# Generate GT labels for all images
# We use a subset for training (up to MAX_TRAIN images) and rest for testing
MAX_TRAIN = 200
IMG_SIZE = 512

random.shuffle(all_image_paths)
train_candidates = all_image_paths[:MAX_TRAIN]
test_candidates  = all_image_paths[MAX_TRAIN:MAX_TRAIN + 50]

gt_records = []  # list of dicts: {img_path, gray_path, points}

for img_path, dataset_name in tqdm(train_candidates, desc='Generating GT labels'):
    gray, vessel_mask, points = generate_gt_label(img_path, IMG_SIZE)
    if points is None or len(points) < 5:
        continue

    stem = Path(img_path).stem
    gray_save_path = f'{OUTPUT_DIR}/gt_labels/{stem}_gray.png'
    label_save_path = f'{OUTPUT_DIR}/gt_labels/{stem}_kps.json'

    cv2.imwrite(gray_save_path, gray)
    with open(label_save_path, 'w') as f:
        json.dump({'points': points, 'dataset': dataset_name, 'img_size': IMG_SIZE}, f)

    gt_records.append({
        'img_path': gray_save_path,
        'label_path': label_save_path,
        'dataset': dataset_name,
        'num_points': len(points)
    })

print(f'\nValid training samples with GT: {len(gt_records)}')
print(f'Average bifurcation points: {np.mean([r["num_points"] for r in gt_records]):.1f}')

## 4. Dataset & DataLoader

In [ ]:
class FundusBifurcationDataset(Dataset):
    def __init__(self, records, img_size=512, augment=True):
        self.records = records
        self.img_size = img_size
        self.augment = augment

    def __len__(self):
        return len(self.records)

    def _make_heatmap(self, points, sigma=3):
        """Convert keypoint list to Gaussian heatmap."""
        heatmap = np.zeros((self.img_size, self.img_size), dtype=np.float32)
        for (x, y) in points:
            x, y = int(x), int(y)
            if 0 <= x < self.img_size and 0 <= y < self.img_size:
                heatmap[y, x] = 1.0
        # Apply Gaussian blur to create soft heatmap
        heatmap = cv2.GaussianBlur(heatmap, (sigma*6+1, sigma*6+1), sigma)
        if heatmap.max() > 0:
            heatmap /= heatmap.max()
        return heatmap

    def _make_point_map(self, points):
        """Binary map with 1 at each keypoint location."""
        pt_map = np.zeros((self.img_size, self.img_size), dtype=np.float32)
        for (x, y) in points:
            x, y = int(x), int(y)
            if 0 <= x < self.img_size and 0 <= y < self.img_size:
                pt_map[y, x] = 1.0
        return pt_map

    def __getitem__(self, idx):
        record = self.records[idx]
        img = cv2.imread(record['img_path'], cv2.IMREAD_GRAYSCALE)
        if img is None or img.shape[0] != self.img_size:
            img = np.zeros((self.img_size, self.img_size), dtype=np.uint8)

        with open(record['label_path']) as f:
            label_data = json.load(f)
        points = label_data['points']

        # Basic augmentation
        if self.augment:
            if random.random() > 0.5:
                img = cv2.flip(img, 1)
                points = [(self.img_size - 1 - x, y) for (x, y) in points]
            if random.random() > 0.5:
                img = cv2.flip(img, 0)
                points = [(x, self.img_size - 1 - y) for (x, y) in points]

        img_tensor = torch.from_numpy(img.astype(np.float32) / 255.0).unsqueeze(0)  # (1, H, W)
        heatmap = self._make_heatmap(points, sigma=3)
        point_map = self._make_point_map(points)

        heatmap_tensor = torch.from_numpy(heatmap).unsqueeze(0)     # (1, H, W)
        point_map_tensor = torch.from_numpy(point_map).unsqueeze(0) # (1, H, W)

        return img_tensor, heatmap_tensor, point_map_tensor


# Train / val split
val_size = max(1, len(gt_records) // 10)
train_records = gt_records[val_size:]
val_records   = gt_records[:val_size]

train_dataset = FundusBifurcationDataset(train_records, IMG_SIZE, augment=True)
val_dataset   = FundusBifurcationDataset(val_records,   IMG_SIZE, augment=False)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_dataset)}  Val: {len(val_dataset)}')

## 5. SuperRetina Model (Detector Head Only)

We use only the detector (U-Net decoder) branch of SuperRetina for bifurcation heatmap regression.

In [ ]:
def double_conv(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, 3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    )


class SuperRetinaDetector(nn.Module):
    """
    Detector branch of SuperRetina adapted for bifurcation heatmap prediction.
    Architecture: shared encoder (VGG-like) + U-Net decoder.
    Input: (B, 1, H, W) grayscale fundus image
    Output: (B, 1, H, W) keypoint probability heatmap in [0, 1]
    """
    def __init__(self):
        super().__init__()
        c1, c2, c3, c4 = 64, 64, 128, 128

        # Encoder (matches SuperRetina shared encoder)
        self.relu = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool2d(2, 2)

        self.conv1a = nn.Conv2d(1,  c1, 3, 1, 1)
        self.conv1b = nn.Conv2d(c1, c1, 3, 1, 1)
        self.conv2a = nn.Conv2d(c1, c2, 3, 1, 1)
        self.conv2b = nn.Conv2d(c2, c2, 3, 1, 1)
        self.conv3a = nn.Conv2d(c2, c3, 3, 1, 1)
        self.conv3b = nn.Conv2d(c3, c3, 3, 1, 1)
        self.conv4a = nn.Conv2d(c3, c4, 3, 1, 1)
        self.conv4b = nn.Conv2d(c4, c4, 3, 1, 1)

        # Detector decoder (U-Net upsampling path)
        self.upsample  = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dconv_up3 = double_conv(c3 + c4, c3)
        self.dconv_up2 = double_conv(c2 + c3, c2)
        self.dconv_up1 = double_conv(c1 + c2, c1)
        self.conv_last = nn.Conv2d(c1, 1, 1)

    def forward(self, x):
        # Encoder
        x = self.relu(self.conv1a(x))
        conv1 = self.relu(self.conv1b(x))
        x = self.pool(conv1)

        x = self.relu(self.conv2a(x))
        conv2 = self.relu(self.conv2b(x))
        x = self.pool(conv2)

        x = self.relu(self.conv3a(x))
        conv3 = self.relu(self.conv3b(x))
        x = self.pool(conv3)

        x = self.relu(self.conv4a(x))
        x = self.relu(self.conv4b(x))

        # Decoder
        x = self.upsample(x)
        x = torch.cat([x, conv3], dim=1)
        x = self.dconv_up3(x)

        x = self.upsample(x)
        x = torch.cat([x, conv2], dim=1)
        x = self.dconv_up2(x)

        x = self.upsample(x)
        x = torch.cat([x, conv1], dim=1)
        x = self.dconv_up1(x)

        out = torch.sigmoid(self.conv_last(x))
        return out


model = SuperRetinaDetector().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}')

## 6. Loss Functions

In [ ]:
class DiceBCELoss(nn.Module):
    """Combined Dice + BCE loss for heatmap regression."""
    def __init__(self, smooth=1.0, bce_weight=0.5):
        super().__init__()
        self.smooth = smooth
        self.bce_weight = bce_weight
        self.bce = nn.BCELoss()

    def dice_loss(self, pred, target):
        pred_flat   = pred.view(-1)
        target_flat = target.view(-1)
        intersection = (pred_flat * target_flat).sum()
        return 1 - (2. * intersection + self.smooth) / (pred_flat.sum() + target_flat.sum() + self.smooth)

    def forward(self, pred, target):
        bce  = self.bce(pred, target)
        dice = self.dice_loss(pred, target)
        return self.bce_weight * bce + (1 - self.bce_weight) * dice


criterion = DiceBCELoss().to(DEVICE)
print('Loss function ready.')

## 7. Training

In [ ]:
NUM_EPOCHS = 50
LR         = 1e-3
PATIENCE   = 10  # early stopping

optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-5)

train_losses = []
val_losses   = []
best_val_loss = float('inf')
patience_counter = 0
CKPT_PATH = f'{OUTPUT_DIR}/checkpoints/best_model.pth'


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, heatmaps, _ in loader:
            imgs     = imgs.to(DEVICE)
            heatmaps = heatmaps.to(DEVICE)
            preds = model(imgs)
            loss  = criterion(preds, heatmaps)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
    return total_loss / len(loader.dataset)


print(f'Starting training for {NUM_EPOCHS} epochs...')
for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss = run_epoch(train_loader, train=True)
    vl_loss = run_epoch(val_loader,   train=False)
    scheduler.step()

    train_losses.append(tr_loss)
    val_losses.append(vl_loss)

    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        torch.save(model.state_dict(), CKPT_PATH)
        patience_counter = 0
        tag = '*'
    else:
        patience_counter += 1
        tag = ''

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{NUM_EPOCHS}  train={tr_loss:.4f}  val={vl_loss:.4f}  {tag}')

    if patience_counter >= PATIENCE:
        print(f'Early stopping at epoch {epoch}.')
        break

print(f'Best val loss: {best_val_loss:.4f}')

## 8. Learning Curve

In [ ]:
epochs_run = list(range(1, len(train_losses) + 1))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(epochs_run, train_losses, label='Train Loss', linewidth=2)
ax.plot(epochs_run, val_losses,   label='Val Loss',   linewidth=2, linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('DiceBCE Loss')
ax.set_title('Learning Curve — SuperRetina Bifurcation Detector')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/results/learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Inference

In [ ]:
def simple_nms_numpy(heatmap, radius=5, thresh=0.3):
    """Non-maximum suppression on a numpy heatmap."""
    from skimage.feature import peak_local_max
    peaks = peak_local_max(heatmap, min_distance=radius, threshold_abs=thresh)
    return peaks[:, ::-1]  # (x, y)


def predict_keypoints(model, img_gray, device, nms_radius=5, nms_thresh=0.3):
    """Run model and extract keypoints via NMS."""
    model.eval()
    h, w = img_gray.shape
    inp = torch.from_numpy(img_gray.astype(np.float32) / 255.0).unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(inp)[0, 0].cpu().numpy()  # (H, W)
    keypoints = simple_nms_numpy(pred, radius=nms_radius, thresh=nms_thresh)
    return pred, keypoints  # heatmap, (N, 2) xy


# Load best checkpoint
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
print('Best model loaded.')

In [ ]:
# Run inference on test images and visualize results
NUM_VIZ = min(6, len(test_candidates))
fig, axes = plt.subplots(NUM_VIZ, 3, figsize=(13, 4 * NUM_VIZ))
if NUM_VIZ == 1:
    axes = [axes]

for row, (img_path, dataset_name) in enumerate(test_candidates[:NUM_VIZ]):
    img_bgr = cv2.imread(img_path)
    gray = preprocess_fundus(img_bgr, IMG_SIZE)
    if gray is None:
        continue
    heatmap_pred, keypoints = predict_keypoints(model, gray, DEVICE)

    # Overlay
    overlay = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    for (kx, ky) in keypoints:
        cv2.circle(overlay, (int(kx), int(ky)), 3, (255, 50, 50), -1)

    axes[row][0].imshow(gray, cmap='gray')
    axes[row][0].set_title(f'{dataset_name}\nInput')
    axes[row][1].imshow(heatmap_pred, cmap='hot')
    axes[row][1].set_title(f'Predicted Heatmap')
    axes[row][2].imshow(overlay)
    axes[row][2].set_title(f'Keypoints ({len(keypoints)})')
    for ax in axes[row]:
        ax.axis('off')

plt.suptitle('Inference Results — Bifurcation Keypoints', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/results/inference_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Vessel Network Analysis

For each test image compute:
- **Node number**: detected bifurcation keypoints count
- **Vessel length**: total skeleton length (pixels)
- **Vessel number**: number of vessel segments connecting nodes (edges in graph)

In [ ]:
def build_vessel_graph(skeleton, bifurcation_points, connect_radius=10):
    """
    Build a NetworkX graph where nodes are bifurcation points and
    edges represent vessel segments traced along the skeleton.
    connect_radius: max pixel distance to connect a skeleton endpoint to a bifurcation node.
    """
    G = nx.Graph()
    for i, (x, y) in enumerate(bifurcation_points):
        G.add_node(i, pos=(x, y))

    if len(bifurcation_points) < 2:
        return G

    bif_arr = np.array(bifurcation_points, dtype=np.float32)
    # For each pair of nodes within a reasonable distance, add an edge
    # (simplified: we check skeleton connectivity via distance)
    from scipy.spatial import cKDTree
    tree = cKDTree(bif_arr)
    pairs = tree.query_pairs(r=IMG_SIZE * 0.3)  # connect nodes within 30% of image width

    for (i, j) in pairs:
        xi, yi = int(bif_arr[i, 0]), int(bif_arr[i, 1])
        xj, yj = int(bif_arr[j, 0]), int(bif_arr[j, 1])
        # Check if there is a skeleton path roughly between these two nodes
        mid_x, mid_y = (xi + xj) // 2, (yi + yj) // 2
        # Sample several points along the line
        num_samples = 10
        xs = np.linspace(xi, xj, num_samples, dtype=int)
        ys = np.linspace(yi, yj, num_samples, dtype=int)
        xs = np.clip(xs, 0, skeleton.shape[1] - 1)
        ys = np.clip(ys, 0, skeleton.shape[0] - 1)
        coverage = skeleton[ys, xs].mean()
        if coverage > 0.2:  # at least 20% of sampled points on skeleton
            dist = np.sqrt((xi - xj)**2 + (yi - yj)**2)
            G.add_edge(i, j, weight=dist)

    return G


def analyze_vessel_network(img_path, model, device, img_size=512, nms_radius=5, nms_thresh=0.3):
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        return None
    gray = preprocess_fundus(img_bgr, img_size)
    if gray is None:
        return None

    # Keypoint detection
    heatmap_pred, keypoints = predict_keypoints(model, gray, device, nms_radius, nms_thresh)

    # Vessel segmentation & skeleton for length/segment analysis
    vessel_mask = segment_vessels_frangi(gray)
    skeleton = skeletonize(vessel_mask > 0).astype(np.uint8)

    # Metrics
    node_number  = len(keypoints)
    vessel_length = int(skeleton.sum())  # pixels

    G = build_vessel_graph(skeleton, keypoints)
    vessel_number = G.number_of_edges()

    return {
        'node_number':   node_number,
        'vessel_length': vessel_length,
        'vessel_number': vessel_number,
        'gray':    gray,
        'skeleton':      skeleton,
        'keypoints':     keypoints,
        'heatmap':       heatmap_pred,
        'graph':         G,
    }


print('Vessel network analysis functions defined.')

In [ ]:
from scipy.spatial import cKDTree

analysis_results = []
NUM_ANALYSIS = min(30, len(test_candidates))

for img_path, dataset_name in tqdm(test_candidates[:NUM_ANALYSIS], desc='Vessel network analysis'):
    result = analyze_vessel_network(img_path, model, DEVICE, IMG_SIZE)
    if result is None:
        continue
    result['img_path'] = img_path
    result['dataset']  = dataset_name
    analysis_results.append(result)

print(f'Analyzed {len(analysis_results)} images.')

In [ ]:
# Summary statistics table
import pandas as pd

rows = [{
    'Dataset':        r['dataset'],
    'Image':          Path(r['img_path']).name,
    'Node Number':    r['node_number'],
    'Vessel Length':  r['vessel_length'],
    'Vessel Number':  r['vessel_number'],
} for r in analysis_results]

df = pd.DataFrame(rows)
print(df.to_string(index=False))

print('\n--- Summary Statistics ---')
print(df[['Node Number', 'Vessel Length', 'Vessel Number']].describe().round(1))

df.to_csv(f'{OUTPUT_DIR}/results/vessel_network_analysis.csv', index=False)
print(f'Saved to {OUTPUT_DIR}/results/vessel_network_analysis.csv')

In [ ]:
# Bar chart of per-dataset averages
if len(df) > 0:
    grp = df.groupby('Dataset')[['Node Number', 'Vessel Length', 'Vessel Number']].mean().reset_index()

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    metrics = ['Node Number', 'Vessel Length', 'Vessel Number']
    colors  = ['steelblue', 'darkorange', 'seagreen']

    for ax, metric, color in zip(axes, metrics, colors):
        ax.bar(grp['Dataset'], grp[metric], color=color, edgecolor='black', linewidth=0.5)
        ax.set_title(f'Avg {metric} per Dataset')
        ax.set_xlabel('Dataset')
        ax.set_ylabel(metric)
        ax.tick_params(axis='x', rotation=30)
        ax.grid(axis='y', alpha=0.3)

    plt.suptitle('Vessel Network Analysis Results', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/results/vessel_analysis_chart.png', dpi=150, bbox_inches='tight')
    plt.show()

## 11. Representative Overlay Visualizations

In [ ]:
NUM_OVERLAY = min(6, len(analysis_results))
fig, axes = plt.subplots(NUM_OVERLAY, 4, figsize=(17, 4 * NUM_OVERLAY))
if NUM_OVERLAY == 1:
    axes = [axes]

for row, result in enumerate(analysis_results[:NUM_OVERLAY]):
    gray      = result['gray']
    skeleton  = result['skeleton']
    keypoints = result['keypoints']
    G         = result['graph']

    # Overlay keypoints on image
    overlay_kp = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    for (kx, ky) in keypoints:
        cv2.circle(overlay_kp, (int(kx), int(ky)), 4, (255, 60, 60), -1)

    # Overlay graph edges on image
    overlay_graph = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    positions = nx.get_node_attributes(G, 'pos')
    for (i, j) in G.edges():
        p1 = (int(positions[i][0]), int(positions[i][1]))
        p2 = (int(positions[j][0]), int(positions[j][1]))
        cv2.line(overlay_graph, p1, p2, (50, 200, 50), 1)
    for (x, y) in keypoints:
        cv2.circle(overlay_graph, (int(x), int(y)), 4, (255, 60, 60), -1)

    name = result['dataset']
    n, l, v = result['node_number'], result['vessel_length'], result['vessel_number']

    axes[row][0].imshow(gray, cmap='gray')
    axes[row][0].set_title(f'{name}\nInput')
    axes[row][1].imshow(skeleton, cmap='gray')
    axes[row][1].set_title('Skeleton')
    axes[row][2].imshow(overlay_kp)
    axes[row][2].set_title(f'Bifurcations\nNodes={n}')
    axes[row][3].imshow(overlay_graph)
    axes[row][3].set_title(f'Vessel Graph\nLen={l} | Segs={v}')
    for ax in axes[row]:
        ax.axis('off')

plt.suptitle('Representative Overlay Results', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/results/overlay_visualizations.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Save All Outputs to Drive

In [ ]:
import shutil

DRIVE_OUTPUT = f'{DRIVE_ROOT}/superretina_output'
if os.path.exists(DRIVE_OUTPUT):
    shutil.rmtree(DRIVE_OUTPUT)
shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT)
print(f'All outputs saved to {DRIVE_OUTPUT}')

---
## Summary of Outputs

| File | Description |
|---|---|
| `results/pipeline_sample.png` | GT generation pipeline visualization |
| `results/learning_curve.png` | Training/validation loss curves |
| `results/inference_results.png` | Predicted keypoints on test images |
| `results/overlay_visualizations.png` | Skeleton + graph overlays |
| `results/vessel_analysis_chart.png` | Per-dataset network metric bar charts |
| `results/vessel_network_analysis.csv` | Per-image node/length/segment counts |
| `checkpoints/best_model.pth` | Best model weights |